# CARS-FLY Phase A - ImageNet-R train-only feasibility

Run every cell in order on a Colab GPU. This notebook extracts **training features only**, selects all hyperparameters on a deterministic training-validation split, and keeps the held-out feature cache absent. It never evaluates ImageNet-R test accuracy. Return the final ZIP for audit before any held-out run.

In [ ]:
# === Edit paths only. Do not edit seed, search grid, model, or gates. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/cars-fly'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/imagenetr_train_feature_cache_seed2025'
TRAIN_CACHE_DIR = '/content/imagenetr_train_feature_cache_seed2025'
OUTPUT_DIR = f'{DRIVE_ROOT}/cars_fly_imagenetr_phasea_seed2025'
CHECKPOINT_SOURCE = 'huggingface'  # or 'google_drive'
DRIVE_CHECKPOINT_PATH = f'{DRIVE_ROOT}/model.safetensors'
BATCH_SIZE = 128
SEED = 2025
CHECKPOINT_SIZE = 346284714
CHECKPOINT_SHA256 = '32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
CONFIG_SHA256 = 'fd9b117751280aa3369f5db7408448cca4e9e90e82d41c4bfb94beb30e95508c'


In [ ]:
# Runtime setup. The initial chdir prevents the deleted-working-directory Colab failure.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo_path = Path(WORK_DIR)
if repo_path.exists():
    shutil.rmtree(repo_path)
subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt', 'kagglehub', 'huggingface_hub'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/cars_fly_imagenetr_train_only.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256, 'Locked config identity mismatch.'
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked seed:', SEED, '| config SHA-256:', CONFIG_SHA256)


In [ ]:
# Obtain and verify the exact frozen ViT checkpoint.
if CHECKPOINT_SOURCE == 'huggingface':
    from huggingface_hub import hf_hub_download
    CHECKPOINT_PATH = hf_hub_download('timm/vit_base_patch16_224.augreg2_in21k_ft_in1k', 'model.safetensors')
elif CHECKPOINT_SOURCE == 'google_drive':
    CHECKPOINT_PATH = DRIVE_CHECKPOINT_PATH
else:
    raise ValueError('CHECKPOINT_SOURCE must be huggingface or google_drive')
checkpoint = Path(CHECKPOINT_PATH)
digest = hashlib.sha256(checkpoint.read_bytes()).hexdigest()
assert checkpoint.stat().st_size == CHECKPOINT_SIZE, 'Checkpoint size mismatch.'
assert digest == CHECKPOINT_SHA256, 'Checkpoint SHA-256 mismatch.'
print('checkpoint PASS:', checkpoint)
print('SHA-256:', digest)


In [ ]:
# Download the processed ImageNet-R artifact and resolve its train/test root.
import kagglehub
from torchvision.datasets import ImageFolder
download_root = Path(kagglehub.dataset_download('zaphat206/imagenet-r')).resolve()
directories = [download_root] + [path for path in download_root.rglob('*') if path.is_dir()]
matches = sorted({path.resolve() for path in directories if (path / 'train').is_dir() and (path / 'test').is_dir()})
assert len(matches) == 1, f'Expected exactly one processed train/test root, found: {matches}'
image_root = matches[0]
processed_root = Path('/content/processed_datasets')
loader_link = processed_root / 'imagenet-r'
processed_root.mkdir(parents=True, exist_ok=True)
if loader_link.is_symlink() or loader_link.is_file():
    loader_link.unlink()
elif loader_link.exists():
    shutil.rmtree(loader_link)
loader_link.symlink_to(image_root, target_is_directory=True)
train_index = ImageFolder(image_root / 'train')
test_index = ImageFolder(image_root / 'test')
assert len(train_index.classes) == 200 and len(test_index.classes) == 200
assert train_index.class_to_idx == test_index.class_to_idx
print('dataset root:', image_root)
print('loader resolves:', loader_link.resolve())
print('train/test samples:', len(train_index), len(test_index), '| classes:', len(train_index.classes))
print('class mapping SHA-256:', hashlib.sha256(json.dumps(train_index.class_to_idx, sort_keys=True).encode()).hexdigest())


In [ ]:
# Mathematical, learner-state, and train-only runner tests.
tests = ['tests/test_cars_fly_math.py', 'tests/test_cars_fly_learner.py', 'tests/test_cars_fly_phasea.py']
subprocess.run([sys.executable, '-m', 'pytest', '-q', *tests], check=True)
print('CARS-FLY correctness gate: PASS')


In [ ]:
# Restore or extract TRAIN embeddings only. Extraction prints tqdm batches and one line per task.
local_cache = Path(TRAIN_CACHE_DIR)
drive_cache = Path(DRIVE_TRAIN_CACHE)
if not (local_cache / 'metadata.json').is_file():
    if (drive_cache / 'metadata.json').is_file():
        print('Restoring train-only cache from Drive...', flush=True)
        local_cache.mkdir(parents=True, exist_ok=True)
        for source in sorted(drive_cache.iterdir()):
            print('COPY', source.name, flush=True)
            shutil.copy2(source, local_cache / source.name)
    else:
        command = [sys.executable, '-u', 'tools/experiment_runner.py', '--extract-features-only', '--extract-train-only', '--root', str(processed_root), '--backbone-checkpoint', CHECKPOINT_PATH, '--backbone-checkpoint-size', str(CHECKPOINT_SIZE), '--backbone-checkpoint-sha256', CHECKPOINT_SHA256, '--feature-cache-dir', TRAIN_CACHE_DIR, '--output-dir', '/content/cars_fly_imagenetr_extract', '--dataset', 'ImageNet-R', '--model-name', 'vit_base_patch16_224', '--data-augmentation', 'vit', '--seed', str(SEED), '--num-classes', '200', '--num-tasks', '20', '--device', 'cuda', '--batch-size', str(BATCH_SIZE), '--num-workers', '2']
        print('Starting 20-task TRAIN-only ViT extraction. Follow tqdm and task lines below.', flush=True)
        subprocess.run(command, check=True)
        assert not (local_cache / 'test.pt').exists()
        if drive_cache.exists():
            raise RuntimeError('Incomplete Drive cache exists; inspect it instead of overwriting.')
        drive_cache.mkdir(parents=True)
        for source in sorted(local_cache.iterdir()):
            print('SAVE', source.name, 'to Drive', flush=True)
            shutil.copy2(source, drive_cache / source.name)
metadata = json.loads((local_cache / 'metadata.json').read_text())
assert metadata['dataset'] == 'ImageNet-R' and metadata['checkpoint_sha256'] == CHECKPOINT_SHA256
assert metadata['feature_dim'] == 768 and metadata['finite'] is True
assert metadata['test_features_materialized'] is False
assert not (local_cache / 'test.pt').exists(), 'Held-out feature cache must remain absent.'
print('train cache PASS:', metadata['train_shape'], '| test.pt absent')


In [ ]:
# Locked train-only Phase A. Live ANCHOR/START/TASK/DONE lines show progress.
output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)
shutil.copy2(config_path, output_path / 'locked_config.json')
command = [sys.executable, '-u', 'tools/cars_fly_phasea.py', '--config', str(config_path), '--feature-cache-dir', TRAIN_CACHE_DIR, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting CARS-FLY Phase A: 16 CARS candidates plus locked controls.', flush=True)
print('A long TASK line means an analytic solve is running; wait for DONE.', flush=True)
started = time.time()
completed = subprocess.run(command)
print(f'Runner elapsed: {(time.time() - started) / 60:.1f} minutes', flush=True)
assert completed.returncode == 0, 'Phase A failed; return the complete traceback without editing config.'
assert (output_path / 'phasea_results.json').is_file()
assert not (local_cache / 'test.pt').exists()
print('CARS-FLY Phase A process: COMPLETE')


In [ ]:
# Compact train-validation result and evidence bundle. STOP after this cell.
import pandas as pd
result = json.loads((output_path / 'phasea_results.json').read_text())
rows = []
for method, item in result['controls'].items():
    diagnostics = item.get('diagnostics', [])
    residuals = [float(value.get('solver_relative_residual_max') or 0.0) for value in diagnostics]
    ranks = [int(value.get('effective_rank') or 0) for value in diagnostics]
    rows.append({'method': method, 'validation_AA': item['validation_average_accuracy'], 'persistent_state_bytes': item['persistent_state_bytes'], 'final_rank': ranks[-1] if ranks else None, 'max_solver_residual': max(residuals, default=None)})
display(pd.DataFrame(rows).sort_values('validation_AA', ascending=False))
print('decision:', result['decision'])
print('uses_test_set:', result['uses_test_set'], '| held_out_test_authorized:', result['held_out_test_authorized'])
print('selected CARS config:', json.dumps(result['selected_cars_config'], indent=2))
print('selected rank schedule:', result['selected_rank_schedule'])
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/cars_fly_imagenetr_phasea_train_only', 'zip', root_dir=OUTPUT_DIR)
artifact_sha = hashlib.sha256(Path(archive).read_bytes()).hexdigest()
print('artifact SHA-256:', artifact_sha)
from google.colab import files
files.download(archive)
print('STOP. Send the ZIP back for audit. Do not evaluate ImageNet-R test yet.')
